In [3]:
class Node:
    def __init__(self, state, empty_position, cost_from_start, heuristic_cost, sequence_score=0, position_score=0, parent=None):
        self.state = state
        self.empty_pos = empty_position
        self.g = cost_from_start
        self.h = heuristic_cost
        self.f = cost_from_start + heuristic_cost
        self.sequence_score = sequence_score
        self.position_score = position_score
        self.parent = parent
    
    def __lt__(self, other):
        return self.f < other.f

# Calculate the # of tiles in the wrong position or the misplaced tiles
def calculate_misplaced(state, goal):
    return sum(1 for i in range(3) for j in range(3) if state[i][j] != goal[i][j] and state[i][j] != 0)

# # Calculate Manhattan distance
def calculate_manhattan(state, goal):
    total_distance = 0
    for row in range(3):
        for col in range(3):
            tile = state[row][col]
            if tile != 0:
                goal_position = find_tile_position(tile, goal)
                if goal_position:
                    total_distance += calculate_distance(row, col, goal_position)
    return total_distance

# Calculate Nilsson's sequence score
def find_tile_position(tile, goal):
    for row in range(3):
        if tile in goal[row]:
            col = goal[row].index(tile)
            return (row, col)
    return None


def calculate_distance(current_row, current_col, goal_position):
    goal_row, goal_col = goal_position
    return abs(goal_row - current_row) + abs(goal_col - current_col)


def calculate_nilsson(state, goal):
    manhattan_cost = calculate_manhattan(state, goal)
    sequence_score = calculate_tile_sequence_score(state)
    return manhattan_cost + 3 * sequence_score


def calculate_tile_sequence_score(state):
    sequence_score = 0
    sequence_goal = [1, 2, 3, 8, 0, 4, 7, 6, 5]

    for i in range(3):
        for j in range(3):
            current_tile = state[i][j]
            idx = i * 3 + j
            if current_tile != 0:
                sequence_score += calculate_tile_score(current_tile, idx, sequence_goal)

    return sequence_score


def calculate_tile_score(tile, idx, sequence_goal):
    score = 0
    if (idx > 0 and tile != sequence_goal[idx - 1]) or (idx < 8 and tile != sequence_goal[idx + 1]):
        score += 2
    if idx == 4:  # Center tile has an additional score
        score += 1
    return score

# Get all valid moves
def get_neighbors(state, empty_pos):
    neighbors = []
    row, col = empty_pos
    moves = [(row + 1, col), (row - 1, col), (row, col + 1), (row, col - 1)]

    for new_row, new_col in moves:
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_state = [list(r) for r in state]  # Create a new state
            new_state[row][col], new_state[new_row][new_col] = new_state[new_row][new_col], new_state[row][col]
            neighbors.append((new_state, (new_row, new_col)))
    return neighbors

# Read input from the file
def read_input_file(filename):
    with open(filename, 'r') as file:
        content = file.read().strip()
        parts = content.split(';')
        if len(parts) != 2:
            raise ValueError("Input file must contain start state and goal separated by a semicolon.")
        
        start_flat = list(map(int, parts[0].strip().split()[1:]))
        goal_flat = list(map(int, parts[1].strip().split()[1:]))

        start = [start_flat[i:i + 3] for i in range(0, 9, 3)]
        goal = [goal_flat[i:i + 3] for i in range(0, 9, 3)]

        return start, goal

# A* Search Algorithm
# retuns None when there is no solution
# retuns the final state that matches the goal (a Node object)
def a_star(start, goal, heuristic):
    empty_pos = next((i, j) for i in range(3) for j in range(3) if start[i][j] == 0)
    initial_h = heuristic(start, goal)
    initial_sequence_score = calculate_tile_sequence_score(start)

    initial_node = Node(start, empty_pos, 0, initial_h, initial_sequence_score, 0, None)
    open_list = [initial_node]  # List of nodes to be expanded
    open_dict = {tuple(map(tuple, start)): initial_node}  # Map from state to node in OPEN
    closed_dict = {}  # Map from state to node in CLOSED
    nodes_generated = 0

    while open_list:
        open_list.sort()  # Sort based on f values
        current_node = open_list.pop(0)
        current_state_tuple = tuple(map(tuple, current_node.state))
        del open_dict[current_state_tuple]  # Remove from OPEN
        closed_dict[current_state_tuple] = current_node  # Add to CLOSED

        nodes_generated += 1

        if current_node.state == goal:
            return current_node, nodes_generated

        for neighbor_state, neighbor_empty_pos in get_neighbors(current_node.state, current_node.empty_pos):
            neighbor_state_tuple = tuple(map(tuple, neighbor_state))

            g = current_node.g + 1
            h = heuristic(neighbor_state, goal)
            f = g + h
            sequence_score = calculate_tile_sequence_score(neighbor_state)

            # Check if neighbor is in OPEN or CLOSED
            if neighbor_state_tuple not in open_dict and neighbor_state_tuple not in closed_dict:
                # Not in OPEN or CLOSED, so add it to OPEN
                neighbor_node = Node(neighbor_state, neighbor_empty_pos, g, h, sequence_score, 0, current_node)
                open_list.append(neighbor_node)
                open_dict[neighbor_state_tuple] = neighbor_node
            else:
                # Already in OPEN or CLOSED
                if neighbor_state_tuple in open_dict:
                    existing_node = open_dict[neighbor_state_tuple]
                else:
                    existing_node = closed_dict[neighbor_state_tuple]

                if g < existing_node.g:
                    # Better path found
                    existing_node.g = g
                    existing_node.h = h
                    existing_node.f = f
                    existing_node.sequence_score = sequence_score
                    existing_node.parent = current_node

                    if neighbor_state_tuple in closed_dict:
                        # Move from CLOSED to OPEN
                        del closed_dict[neighbor_state_tuple]
                        open_list.append(existing_node)
                        open_dict[neighbor_state_tuple] = existing_node

    return None, nodes_generated

# Print all nodes from the start state to the goal
def print_solution(node, nodes_generated, heuristic):
    if node is None:
        print("No solution was found.")
        return

    path = []
    while node:
        path.append(node)
        node = node.parent
    path.reverse()

    for step in path:
        for row in step.state:
            print(" ".join(map(str, row)))
        if heuristic == calculate_nilsson:
            print(f"f(n)={step.f}, g(n)={step.g}, h(n)={step.h}, P(n)={step.position_score}, S(n)={step.sequence_score}\n")
        else:
            print(f"f(n)={step.f}, g(n)={step.g}, h(n)={step.h}\n")

    print(f"Search Cost (# of nodes generated): {nodes_generated}\n\n")

# This is the code that runs when the script is executed
if __name__ == "__main__":
    start, goal = read_input_file("astar_in.txt")
    print(f"Start:\n{start}")
    print(f"Goal:\n{goal}\n")

    print("Using Misplaced Tiles heuristic:\n")
    solution_misplaced, nodes_misplaced = a_star(start, goal, calculate_misplaced)
    print_solution(solution_misplaced, nodes_misplaced, calculate_misplaced)

    print("\nUsing Manhattan Distance heuristic:\n")
    solution_manhattan, nodes_manhattan = a_star(start, goal, calculate_manhattan)
    print_solution(solution_manhattan, nodes_manhattan, calculate_manhattan)

    print("\nUsing Nilsson's Sequence Score heuristic:\n")
    solution_nilsson, nodes_nilsson = a_star(start, goal, calculate_nilsson)
    print_solution(solution_nilsson, nodes_nilsson, calculate_nilsson)

Start:
[[2, 1, 6], [4, 0, 8], [7, 5, 3]]
Goal:
[[1, 2, 3], [8, 0, 4], [7, 6, 5]]

Using Misplaced Tiles heuristic:

2 1 6
4 0 8
7 5 3
f(n)=7, g(n)=0, h(n)=7

2 1 6
4 8 0
7 5 3
f(n)=8, g(n)=1, h(n)=7

2 1 0
4 8 6
7 5 3
f(n)=9, g(n)=2, h(n)=7

2 0 1
4 8 6
7 5 3
f(n)=10, g(n)=3, h(n)=7

2 8 1
4 0 6
7 5 3
f(n)=11, g(n)=4, h(n)=7

2 8 1
4 6 0
7 5 3
f(n)=12, g(n)=5, h(n)=7

2 8 1
4 6 3
7 5 0
f(n)=13, g(n)=6, h(n)=7

2 8 1
4 6 3
7 0 5
f(n)=13, g(n)=7, h(n)=6

2 8 1
4 0 3
7 6 5
f(n)=13, g(n)=8, h(n)=5

2 8 1
0 4 3
7 6 5
f(n)=14, g(n)=9, h(n)=5

0 8 1
2 4 3
7 6 5
f(n)=15, g(n)=10, h(n)=5

8 0 1
2 4 3
7 6 5
f(n)=16, g(n)=11, h(n)=5

8 1 0
2 4 3
7 6 5
f(n)=17, g(n)=12, h(n)=5

8 1 3
2 4 0
7 6 5
f(n)=17, g(n)=13, h(n)=4

8 1 3
2 0 4
7 6 5
f(n)=17, g(n)=14, h(n)=3

8 1 3
0 2 4
7 6 5
f(n)=18, g(n)=15, h(n)=3

0 1 3
8 2 4
7 6 5
f(n)=18, g(n)=16, h(n)=2

1 0 3
8 2 4
7 6 5
f(n)=18, g(n)=17, h(n)=1

1 2 3
8 0 4
7 6 5
f(n)=18, g(n)=18, h(n)=0

Search Cost (# of nodes generated): 1747



Using Manhattan D